# Update the finalized list of sample labs - wave 2

In [1]:
# Set up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[2] # Adjust this path to root of repo
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import load_workbook
from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import Font, PatternFill
import os
import warnings

# Suppress irrelevant conditional formatting warnings when reading Excel files
warnings.filterwarnings("ignore", message="Conditional Formatting extension is not supported")

In [2]:
# Load datasets
existing_assignments = pd.read_csv(config.WAVE2_ENUMERATORS / "assignedlabs.csv")
all_enumerators = pd.read_excel(config.WAVE2_ENUMERATORS / "all_enumerators.xlsx")
correct_assignments = pd.read_excel(config.WAVE2_ENUMERATORS / "correct_assignment.xlsx")

In [3]:
# Merge to get enum_foldername
assignments = existing_assignments.merge(
    all_enumerators[["enum_id", "foldername", "research team"]], 
    on="enum_id", how="left"
)

In [4]:
# Create file containing all checked labs
checked_all = []

for enum_id, enum_data in assignments.groupby("enum_id"):

    # Get enumerator info
    id = enum_data["enum_id"].iloc[0]
    name = enum_data["foldername"].iloc[0]
    folder_name = f"{name}_data"
    research_team = enum_data["research team"].iloc[0]

    # Find all relevant excel files (i.e. lab_assignment or lab_assignment_2026_...)
    if research_team == 1:
        excel_files = [f for f in os.listdir(os.path.join(config.SWITCHDRIVE_ROOT, folder_name)) 
                   if f.startswith("lab_assignment_2026")]
    else:
        excel_files = [f for f in os.listdir(os.path.join(config.SWITCHDRIVE_ROOT, folder_name)) 
                   if f.startswith("lab_assignment_2026") or f == "lab_assignment.xlsx"]
    
    # Load the files as dataframe
    for excel_file in excel_files:
        file_path = os.path.join(config.SWITCHDRIVE_ROOT, folder_name, excel_file)
        checked_file = pd.read_excel(file_path, sheet_name="Lab Assignments")
        checked_file ["enum_id"] = id
        checked_all.append(checked_file[[
            "labgroupid", "Faculty", 
            "Contact person (if different to prof)", 
            "Contact email (if different)", 
            "Comments", 
            "enum_id"
        ]])

# Combine all checked files
if checked_all:  # make sure the list is not empty
    combined_checked = pd.concat(checked_all, ignore_index=True)

In [5]:
# Correct assignments 

# replace contact person and email with the correct ones from the correct_assignments file
combined_checked = combined_checked.set_index("labgroupid")
combined_checked.update(
    correct_assignments.set_index("labgroupid")[[
        "enum_id", "Contact person (if different to prof)", "Contact email (if different)"
    ]]
)
combined_checked = combined_checked.reset_index()

In [6]:
# Merge with assignments on labgroupid and enum_id
updated_assignments = assignments.merge(
    combined_checked,
    on=["labgroupid", "enum_id"],
    how="left",
    suffixes=('', '_checked')
)

In [8]:
# Check how many labs in our sample (out_of_sample = 0)
num_visited = updated_assignments[updated_assignments["out_of_sample"] == 0].shape[0]
print(f"{num_visited} labs are in sample.")
# Check how many treatment and control labs in our sample
num_treatment_visited = updated_assignments[(updated_assignments["out_of_sample"] == 0) &
                                            (updated_assignments["Treatment Status"] == "treatment")].shape[0]
num_control_visited = updated_assignments[(updated_assignments["out_of_sample"] == 0) &
                                            (updated_assignments["Treatment Status"] == "control")].shape[0]
print(f"{num_treatment_visited} treatment labs are in sample.")
print(f"{num_control_visited} control labs are in sample.")

8 labs are in sample.
5 treatment labs are in sample.
3 control labs are in sample.


In [9]:
# Partition into sample and out of sample
sample_labs = updated_assignments[updated_assignments["out_of_sample"] == 0]
out_of_sample_labs = updated_assignments[updated_assignments["out_of_sample"] == 1]

In [10]:
# Save the sample labs
sample_labs.to_csv(config.WAVE2_BL_RAW_SAMPLE / "final_sample.csv", index=False)
sample_labs.to_csv(config.WAVE2_BL_RAW_SAMPLE_BACKUP / "final_sample.csv", index=False)

In [11]:
# Save the out of sample labs
out_of_sample_labs.to_csv(config.WAVE2_LABS_LIST / "out_of_sample_labs.csv", index=False)
updated_assignments.to_csv(config.WAVE2_LABS_LIST / "final_all_labs.csv", index=False)